# Limpieza y normalización · venta_idealista.csv

Objetivo: cargar el dataset y construir una cadena de transformaciones **sin modificar el DataFrame original**.

Estrategia:
- `df_raw`: lectura tal cual
- `df_work`, `df_bool_norm`, ...: pasos de limpieza incrementales (copias)


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import csv


## 1) Carga del CSV

Si el CSV tiene comillas rotas o líneas corruptas, usamos `on_bad_lines="warn"` para **no bloquear** el análisis.

In [2]:
df_raw = pd.read_csv(
    "../data/raw/venta_idealista.csv",
    engine="python",
    on_bad_lines="warn",
    encoding="utf-8",
    encoding_errors="replace",
    skipinitialspace=True,
    index_col=0   # ← primera columna como índice
)

df_raw.head()

,fuente,operacion,tipo_inmueble,subtipo,direccion_texto,barrio_zona,municipio,provincia,precio_eur,precio_anterior_eur,...,garaje_incluido,garaje_detalle,habitaciones,superficie_m2,planta,exterior,ascensor,obra_nueva_categoria,obra_nueva_flag,notas_parseo
id_anuncio,,,,,,,,,,,,,,,,,,,,,
1,Inmobiliaria E-Castro,venta,piso,NaN,"Avenida Riomar, 1",Cotolino,Castro-Urdiales,Cantabria,498000,NaN,...,sí,NaN,4,121,4,sí,sí,otros,0,NaN
2,Inmobiliarias.com,venta,casa,chalét independiente,"Barrio Brazomar, 46",Cotolino,Castro-Urdiales,Cantabria,575000,NaN,...,sí,NaN,5,232,NaN,desconocido,desconocido,otros,0,NaN
3,Inmobiliaria E-Castro,venta,piso,NaN,"Calle Leonardo Rucabado, 6",Centro,Castro-Urdiales,Cantabria,299000,NaN,...,desconocido,NaN,2,84,3,sí,sí,otros,0,Garaje no indicado
4,Inmobiliaria E-Castro,venta,piso,NaN,"Calle de la Ronda, 32",Centro,Castro-Urdiales,Cantabria,340000,NaN,...,desconocido,NaN,4,99,5,sí,sí,otros,0,Garaje no indicado
5,Inmobiliaria E-Castro,venta,casa,chalét independiente,Barrio Sámano,Sámano,Castro-Urdiales,Cantabria,799000,NaN,...,sí,NaN,5,511,NaN,desconocido,desconocido,otros,0,Dirección sin número


In [3]:
# Primer DataFrame de trabajo (no modifica el original)
df_work = df_raw.copy()

## 2) Normalización de columnas booleanas

Convertimos `sí/no/desconocido` a `1/0/NaN` y lo dejamos como numérico.


In [26]:
df_bool_norm = df_work.copy()

cols_bool = ["ascensor", "exterior", "garaje_incluido"]

for col in cols_bool:
    if col not in df_bool_norm.columns:
        continue

    df_bool_norm[col] = (
        df_bool_norm[col]
            .astype(str)
            .str.strip()
            .str.lower()
            .replace({
                "desconocido": np.nan,
                "nan": np.nan,
                "sí": 1,
                "si": 1,
                "no": 0
            })
    )
    df_bool_norm[col] = pd.to_numeric(df_bool_norm[col], errors="coerce")
df_bool_norm.head()


,fuente,operacion,tipo_inmueble,subtipo,direccion_texto,barrio_zona,municipio,provincia,precio_eur,precio_anterior_eur,...,garaje_incluido,garaje_detalle,habitaciones,superficie_m2,planta,exterior,ascensor,obra_nueva_categoria,obra_nueva_flag,notas_parseo
id_anuncio,,,,,,,,,,,,,,,,,,,,,
1,Inmobiliaria E-Castro,venta,piso,NaN,"Avenida Riomar, 1",Cotolino,Castro-Urdiales,Cantabria,498000,NaN,...,1.0,NaN,4,121,4,1.0,1.0,otros,0,NaN
2,Inmobiliarias.com,venta,casa,chalét independiente,"Barrio Brazomar, 46",Cotolino,Castro-Urdiales,Cantabria,575000,NaN,...,1.0,NaN,5,232,NaN,NaN,NaN,otros,0,NaN
3,Inmobiliaria E-Castro,venta,piso,NaN,"Calle Leonardo Rucabado, 6",Centro,Castro-Urdiales,Cantabria,299000,NaN,...,NaN,NaN,2,84,3,1.0,1.0,otros,0,Garaje no indicado
4,Inmobiliaria E-Castro,venta,piso,NaN,"Calle de la Ronda, 32",Centro,Castro-Urdiales,Cantabria,340000,NaN,...,NaN,NaN,4,99,5,1.0,1.0,otros,0,Garaje no indicado
5,Inmobiliaria E-Castro,venta,casa,chalét independiente,Barrio Sámano,Sámano,Castro-Urdiales,Cantabria,799000,NaN,...,1.0,NaN,5,511,NaN,NaN,NaN,otros,0,Dirección sin número


## 3) Flags derivados

- `descuento_flag = 1` si existe `descuento_pct`.


In [27]:
df_flags = df_bool_norm.copy()

if "descuento_pct" in df_flags.columns:
    df_flags["descuento_flag"] = df_flags["descuento_pct"].notna().astype(int)

df_flags.head()


,fuente,operacion,tipo_inmueble,subtipo,direccion_texto,barrio_zona,municipio,provincia,precio_eur,precio_anterior_eur,...,garaje_detalle,habitaciones,superficie_m2,planta,exterior,ascensor,obra_nueva_categoria,obra_nueva_flag,notas_parseo,descuento_flag
id_anuncio,,,,,,,,,,,,,,,,,,,,,
1,Inmobiliaria E-Castro,venta,piso,NaN,"Avenida Riomar, 1",Cotolino,Castro-Urdiales,Cantabria,498000,NaN,...,NaN,4,121,4,1.0,1.0,otros,0,NaN,0
2,Inmobiliarias.com,venta,casa,chalét independiente,"Barrio Brazomar, 46",Cotolino,Castro-Urdiales,Cantabria,575000,NaN,...,NaN,5,232,NaN,NaN,NaN,otros,0,NaN,0
3,Inmobiliaria E-Castro,venta,piso,NaN,"Calle Leonardo Rucabado, 6",Centro,Castro-Urdiales,Cantabria,299000,NaN,...,NaN,2,84,3,1.0,1.0,otros,0,Garaje no indicado,0
4,Inmobiliaria E-Castro,venta,piso,NaN,"Calle de la Ronda, 32",Centro,Castro-Urdiales,Cantabria,340000,NaN,...,NaN,4,99,5,1.0,1.0,otros,0,Garaje no indicado,0
5,Inmobiliaria E-Castro,venta,casa,chalét independiente,Barrio Sámano,Sámano,Castro-Urdiales,Cantabria,799000,NaN,...,NaN,5,511,NaN,NaN,NaN,otros,0,Dirección sin número,0


## 4) Limpieza de `planta`

Normalizamos 'bajo' a 0 y convertimos a numérico.


In [ ]:
df_4 = df_flags.copy()

if "planta" in df_4.columns:
    df_4["planta"] = (
        df_4["planta"]
            .astype(str)
            .str.strip()
            .replace({"bajo": "0", "Bajo": "0"})
    )
    df_4["planta"] = pd.to_numeric(df_4["planta"], errors="coerce")

,fuente,operacion,tipo_inmueble,subtipo,direccion_texto,barrio_zona,municipio,provincia,precio_eur,precio_anterior_eur,...,garaje_detalle,habitaciones,superficie_m2,planta,exterior,ascensor,obra_nueva_categoria,obra_nueva_flag,notas_parseo,descuento_flag
id_anuncio,,,,,,,,,,,,,,,,,,,,,
1,Inmobiliaria E-Castro,venta,piso,NaN,"Avenida Riomar, 1",Cotolino,Castro-Urdiales,Cantabria,498000,NaN,...,NaN,4,121,4.0,1.0,1.0,otros,0,NaN,0
2,Inmobiliarias.com,venta,casa,chalét independiente,"Barrio Brazomar, 46",Cotolino,Castro-Urdiales,Cantabria,575000,NaN,...,NaN,5,232,NaN,NaN,NaN,otros,0,NaN,0
3,Inmobiliaria E-Castro,venta,piso,NaN,"Calle Leonardo Rucabado, 6",Centro,Castro-Urdiales,Cantabria,299000,NaN,...,NaN,2,84,3.0,1.0,1.0,otros,0,Garaje no indicado,0
4,Inmobiliaria E-Castro,venta,piso,NaN,"Calle de la Ronda, 32",Centro,Castro-Urdiales,Cantabria,340000,NaN,...,NaN,4,99,5.0,1.0,1.0,otros,0,Garaje no indicado,0
5,Inmobiliaria E-Castro,venta,casa,chalét independiente,Barrio Sámano,Sámano,Castro-Urdiales,Cantabria,799000,NaN,...,NaN,5,511,NaN,NaN,NaN,otros,0,Dirección sin número,0


## 5) Normalización de tipo y subtipo

Pasamos a minúsculas, quitamos espacios y unificamos categorías.


In [7]:
df_5 = df_4.copy()

for col in ["tipo_inmueble", "subtipo"]:
    if col in df_5.columns:
        df_5[col] = df_5[col].astype(str).str.lower().str.strip().replace({"nan": np.nan})


## 6) Filtrado de tipos no relevantes

Eliminamos registros cuyo `tipo_inmueble` no es vivienda (según la lista).


In [8]:
df_6 = df_5.copy()

valores_eliminar = ["finca", "finca_rustica", "garaje", "garaje/local", "piso/casa", "edificio"]

if "tipo_inmueble" in df_6.columns:
    df_6 = df_6[~df_6["tipo_inmueble"].isin(valores_eliminar)].copy()


## 7) Reasignación dúplex/ático a subtipo y `piso`

Si `tipo_inmueble` viene como 'ático' o 'dúplex', lo pasamos a `subtipo` y forzamos `tipo_inmueble = piso`.


In [9]:
df_7 = df_6.copy()

if "tipo_inmueble" in df_7.columns and "subtipo" in df_7.columns:
    mask_duplex_atico = df_7["tipo_inmueble"].isin(["dúplex", "duplex", "ático", "atico"])
    df_7.loc[mask_duplex_atico, "subtipo"] = df_7.loc[mask_duplex_atico, "tipo_inmueble"]
    df_7.loc[mask_duplex_atico, "tipo_inmueble"] = "piso"


## 8) Unificación de categorías (chalet → casa)


In [10]:
df_8 = df_7.copy()

if "tipo_inmueble" in df_8.columns:
    df_8["tipo_inmueble"] = df_8["tipo_inmueble"].replace({"chalet": "casa"})


## 9) Piso sin subtipo explícito

Si es `piso` y no es ático/dúplex, asignamos subtipo por defecto `apartamento`.


In [11]:
df_9 = df_8.copy()

if "tipo_inmueble" in df_9.columns and "subtipo" in df_9.columns:
    mask_piso_sin_subtipo = (
        (df_9["tipo_inmueble"] == "piso") &
        (~df_9["subtipo"].isin(["dúplex", "duplex", "ático", "atico"]))
    )
    df_9.loc[mask_piso_sin_subtipo & df_9["subtipo"].isna(), "subtipo"] = "apartamento"


In [12]:
df_9["subtipo"].unique() if "subtipo" in df_9.columns else None

array(['apartamento', 'chalét independiente', 'ático', 'dúplex',
       'chalét adosado', 'chalét pareado', nan, 'subtipo',
       'casa de piedra', 'casa de pueblo', 'casa independiente', 'casa',
       'chalét', 'estudio', 'masía', 'villa', 'casa montañesa', 'casona',
       'casa rural', 'casa indiana', 'casa unifamiliar', 'casa adosada',
       'palacio', 'torre', 'casa o chalet independiente',
       'chalet pareado', 'casa o chalet independent\ufeffe',
       'chalet adosado', 'chalet', 'obra nueva (promoción)'], dtype=object)

## 10) Mapeo fino de `subtipo`

Unificamos variantes de texto.


In [13]:
df_10 = df_9.copy()

if "subtipo" in df_10.columns:
    df_10["subtipo"] = (
        df_10["subtipo"]
            .astype(str)
            .str.lower()
            .str.strip()
            .replace({"nan": np.nan})
            .replace({
                "chalét independiente": "casa independiente",
                "chalet independiente": "casa independiente",
                "chalét pareado": "casa adosada",
                "chalet pareado": "casa adosada",
                "chalét adosado": "casa adosada",
                "chalet adosado": "casa adosada",
                "casa o chalet independiente": "casa independiente",
                "casa o chalet independiente\ufeff": "casa independiente",
                "chalet": "casa independiente",
                "casa unifamiliar": "casa independiente"
            })
    )


In [14]:
df_10["subtipo"].unique() if "subtipo" in df_10.columns else None

array(['apartamento', 'casa independiente', 'ático', 'dúplex',
       'casa adosada', nan, 'subtipo', 'casa de piedra', 'casa de pueblo',
       'casa', 'chalét', 'estudio', 'masía', 'villa', 'casa montañesa',
       'casona', 'casa rural', 'casa indiana', 'palacio', 'torre',
       'casa o chalet independent\ufeffe', 'obra nueva (promoción)'],
      dtype=object)

## 11) Relleno coherente de `subtipo`

Si falta `subtipo`:
- si `tipo_inmueble` es `casa` → `casa independiente`
- si `tipo_inmueble` es `piso` → `apartamento`


In [15]:
df_11 = df_10.copy()

if "tipo_inmueble" in df_11.columns and "subtipo" in df_11.columns:
    df_11.loc[(df_11["subtipo"].isna()) & (df_11["tipo_inmueble"] == "casa"), "subtipo"] = "casa independiente"
    df_11.loc[(df_11["subtipo"].isna()) & (df_11["tipo_inmueble"] == "piso"), "subtipo"] = "apartamento"


## 12) Limpieza de `direccion_texto`

Elimina comillas sobrantes al inicio/fin.


In [16]:
df_12 = df_11.copy()

if "direccion_texto" in df_12.columns:
    df_12["direccion_texto"] = (
        df_12["direccion_texto"]
            .astype(str)
            .str.strip()
            .str.strip('"')
            .str.strip("'")
            .str.strip()
            .replace({"nan": np.nan})
    )


## 13) Tipado de variables numéricas + filtrado mínimo

Convertimos a numérico y descartamos filas sin `precio_eur`.


In [17]:
df_13 = df_12.copy()

cols_numericas = [
    "precio_eur",
    "precio_anterior_eur",
    "superficie_m2",
    "habitaciones",
    "planta",
    "descuento_pct"
]

for col in cols_numericas:
    if col in df_13.columns:
        df_13[col] = pd.to_numeric(df_13[col], errors="coerce")

for col in cols_numericas:
    if col in df_13.columns:
        print(col, df_13[col].isna().sum())


precio_eur 3
precio_anterior_eur 785
superficie_m2 157
habitaciones 97
planta 497
descuento_pct 786


In [18]:
df_clean = df_13.copy()

if "precio_eur" in df_clean.columns:
    df_clean = df_clean[df_clean["precio_eur"].notna()].copy()

df_clean.head(20)

,fuente,operacion,tipo_inmueble,subtipo,direccion_texto,barrio_zona,municipio,provincia,precio_eur,precio_anterior_eur,...,garaje_detalle,habitaciones,superficie_m2,planta,exterior,ascensor,obra_nueva_categoria,obra_nueva_flag,notas_parseo,descuento_flag
id_anuncio,,,,,,,,,,,,,,,,,,,,,
1,Inmobiliaria E-Castro,venta,piso,apartamento,"Avenida Riomar, 1",Cotolino,Castro-Urdiales,Cantabria,498000.0,NaN,...,NaN,4.0,121.0,4.0,1.0,1.0,otros,0,NaN,0
2,Inmobiliarias.com,venta,casa,casa independiente,"Barrio Brazomar, 46",Cotolino,Castro-Urdiales,Cantabria,575000.0,NaN,...,NaN,5.0,232.0,NaN,NaN,NaN,otros,0,NaN,0
3,Inmobiliaria E-Castro,venta,piso,apartamento,"Calle Leonardo Rucabado, 6",Centro,Castro-Urdiales,Cantabria,299000.0,NaN,...,NaN,2.0,84.0,3.0,1.0,1.0,otros,0,Garaje no indicado,0
4,Inmobiliaria E-Castro,venta,piso,apartamento,"Calle de la Ronda, 32",Centro,Castro-Urdiales,Cantabria,340000.0,NaN,...,NaN,4.0,99.0,5.0,1.0,1.0,otros,0,Garaje no indicado,0
5,Inmobiliaria E-Castro,venta,casa,casa independiente,Barrio Sámano,Sámano,Castro-Urdiales,Cantabria,799000.0,NaN,...,NaN,5.0,511.0,NaN,NaN,NaN,otros,0,Dirección sin número,0
6,Inmobiliarias.com,venta,piso,ático,Calle de Angel Perez Hornoas Cholo,Brazomar,Castro-Urdiales,Cantabria,360000.0,NaN,...,NaN,3.0,130.0,6.0,1.0,1.0,otros,0,NaN,0
7,Inmobiliarias.com,venta,piso,ático,Calle de la Granja,Cotolino,Castro-Urdiales,Cantabria,405000.0,NaN,...,NaN,2.0,90.0,4.0,1.0,1.0,otros,0,NaN,0
8,Inmobiliarias.com,venta,piso,apartamento,Calle Ataúlfo Argenta,Cotolino,Castro-Urdiales,Cantabria,415000.0,NaN,...,NaN,3.0,110.0,2.0,1.0,1.0,otros,0,NaN,0
9,Inmobiliarias.com,venta,casa,casa independiente,Barrio Montealegre,Sámano,Castro-Urdiales,Cantabria,455000.0,NaN,...,NaN,5.0,266.0,NaN,NaN,NaN,otros,0,Garaje no indicado,0


## 14) Checks rápidos

- Tipos de datos
- Ejemplos de filas


In [19]:
df_clean[cols_numericas].dtypes if all(c in df_clean.columns for c in cols_numericas) else df_clean.dtypes

precio_eur             float64
precio_anterior_eur    float64
superficie_m2          float64
habitaciones           float64
planta                 float64
descuento_pct          float64
dtype: object

In [20]:
df_clean.dtypes

fuente                   object
operacion                object
tipo_inmueble            object
subtipo                  object
direccion_texto          object
barrio_zona              object
municipio                object
provincia                object
precio_eur              float64
precio_anterior_eur     float64
descuento_pct           float64
garaje_incluido         float64
garaje_detalle           object
habitaciones            float64
superficie_m2           float64
planta                  float64
exterior                float64
ascensor                float64
obra_nueva_categoria     object
obra_nueva_flag          object
notas_parseo             object
descuento_flag            int32
dtype: object

In [21]:
df_clean.head(20)

,fuente,operacion,tipo_inmueble,subtipo,direccion_texto,barrio_zona,municipio,provincia,precio_eur,precio_anterior_eur,...,garaje_detalle,habitaciones,superficie_m2,planta,exterior,ascensor,obra_nueva_categoria,obra_nueva_flag,notas_parseo,descuento_flag
id_anuncio,,,,,,,,,,,,,,,,,,,,,
1,Inmobiliaria E-Castro,venta,piso,apartamento,"Avenida Riomar, 1",Cotolino,Castro-Urdiales,Cantabria,498000.0,NaN,...,NaN,4.0,121.0,4.0,1.0,1.0,otros,0,NaN,0
2,Inmobiliarias.com,venta,casa,casa independiente,"Barrio Brazomar, 46",Cotolino,Castro-Urdiales,Cantabria,575000.0,NaN,...,NaN,5.0,232.0,NaN,NaN,NaN,otros,0,NaN,0
3,Inmobiliaria E-Castro,venta,piso,apartamento,"Calle Leonardo Rucabado, 6",Centro,Castro-Urdiales,Cantabria,299000.0,NaN,...,NaN,2.0,84.0,3.0,1.0,1.0,otros,0,Garaje no indicado,0
4,Inmobiliaria E-Castro,venta,piso,apartamento,"Calle de la Ronda, 32",Centro,Castro-Urdiales,Cantabria,340000.0,NaN,...,NaN,4.0,99.0,5.0,1.0,1.0,otros,0,Garaje no indicado,0
5,Inmobiliaria E-Castro,venta,casa,casa independiente,Barrio Sámano,Sámano,Castro-Urdiales,Cantabria,799000.0,NaN,...,NaN,5.0,511.0,NaN,NaN,NaN,otros,0,Dirección sin número,0
6,Inmobiliarias.com,venta,piso,ático,Calle de Angel Perez Hornoas Cholo,Brazomar,Castro-Urdiales,Cantabria,360000.0,NaN,...,NaN,3.0,130.0,6.0,1.0,1.0,otros,0,NaN,0
7,Inmobiliarias.com,venta,piso,ático,Calle de la Granja,Cotolino,Castro-Urdiales,Cantabria,405000.0,NaN,...,NaN,2.0,90.0,4.0,1.0,1.0,otros,0,NaN,0
8,Inmobiliarias.com,venta,piso,apartamento,Calle Ataúlfo Argenta,Cotolino,Castro-Urdiales,Cantabria,415000.0,NaN,...,NaN,3.0,110.0,2.0,1.0,1.0,otros,0,NaN,0
9,Inmobiliarias.com,venta,casa,casa independiente,Barrio Montealegre,Sámano,Castro-Urdiales,Cantabria,455000.0,NaN,...,NaN,5.0,266.0,NaN,NaN,NaN,otros,0,Garaje no indicado,0
